# Vector Memory

Vector memory stores conversation information as **embeddings** in a vector
store and retrieves relevant memories based on **semantic similarity**.

Unlike normal conversation memory, which stores the complete history, or
summarization memory, which compresses older messages into a summary, vector
memory retrieves only the information relevant to the current query.

## Basic Flow

```text
Conversation
     ↓
Embedding Model
     ↓
Vector Store
     ↓
User Query
     ↓
Query Embedding
     ↓
Similarity Search
     ↓
Relevant Memories
     ↓
LLM
     ↓
Response
```


In [4]:
# Previous summarization memory notebook ko reuse garna
%run ./memory/18_summarization_memory.ipynb

LLM loaded successfully!
[]
[HumanMessage(content='My name is Arjun.', additional_kwargs={}, response_metadata={})]
Nice to meet you, Arjun!
[HumanMessage(content='My name is Arjun.', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Arjun!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Your name is Arjun!
HumanMessage: My name is Arjun.
AIMessage: Nice to meet you, Arjun!
HumanMessage: What is my name?
AIMessage: Your name is Arjun!
Assistant: Python is a fantastic language! Many people share your enthusiasm for Python, and it's easy to see why. Python is a high-level language that's known for its simplicity, readability, and ease of use. It's a great language for beginners and experienced programmers alike.

What is it about Python that you enjoy the most? Is it the syntax, the vast number of libraries and frameworks, or something else entirely?

Also, what kind of projects or areas of programming do you enjoy work

In [5]:
# Ollama embeddings use garna
from langchain_ollama import OllamaEmbeddings

# Local embedding model load garne
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Embedding model loaded successfully!")

Embedding model loaded successfully!


In [6]:
# Chroma vector store use garna
from langchain_community.vectorstores import Chroma

# Vector memory ko lagi Chroma store create garne
vector_memory = Chroma(
    collection_name="conversation_vector_memory", embedding_function=embeddings
)

print("Vector memory initialized successfully!")

C:\Users\Acer\AppData\Local\Temp\ipykernel_7368\1693629428.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\Acer\AppData\Local\Temp\ipykernel_7368\1693629428.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_memory = Chroma(


Vector memory initialized successfully!


In [7]:
# Conversation message vector memory ma store garne
def store_memory(message):

    # Message ko content ra role store garne
    vector_memory.add_texts(
        texts=[message.content], metadatas=[{"role": message.__class__.__name__}]
    )

In [8]:
# Relevant memories retrieve garne
def retrieve_memories(query, k=3):

    # Query ko semantic similarity ko basis ma memories retrieve garne
    memories = vector_memory.similarity_search(query, k=k)

    return memories

In [9]:
# Vector memory use garera chat garne
def chat_with_vector_memory(user_message, k=3):

    # User query bata relevant memories retrieve garne
    memories = retrieve_memories(user_message, k=k)

    # Retrieved memories lai context ma convert garne
    memory_context = "\n".join(memory.page_content for memory in memories)

    # Retrieved memories use garera prompt create garne
    prompt = f"""
Use the provided memories to answer the user's question.

Memories:
{memory_context}

User question:
{user_message}

Answer:
"""

    # LLM bata response generate garne
    response = llm.invoke(prompt)

    # User message vector memory ma store garne
    store_memory(HumanMessage(content=user_message))

    # Assistant response vector memory ma store garne
    store_memory(AIMessage(content=response.content))

    return response.content

In [10]:
# Testing ko lagi fresh vector memory create garne
vector_memory = Chroma(
    collection_name="conversation_vector_memory_test", embedding_function=embeddings
)

In [11]:
# Important user information store garne
store_memory(HumanMessage(content="My favorite programming language is Python."))

store_memory(HumanMessage(content="I am building a RAG system locally using Ollama."))

store_memory(HumanMessage(content="I want to use FastAPI as the backend."))

In [12]:
# Semantically related memory retrieve garne
memories = retrieve_memories("Which programming language do I prefer?")

for memory in memories:
    print(memory.page_content)

My favorite programming language is Python.
I am building a RAG system locally using Ollama.
I want to use FastAPI as the backend.


In [13]:
# Retrieved memory use garera answer generate garne
response = chat_with_vector_memory("Which programming language do I prefer?")

print("Assistant:", response)

Assistant: According to the memories, your favorite programming language is Python.


In [14]:
# Different wording use garera semantic retrieval test garne
response = chat_with_vector_memory("What language do I like for programming?")

print("Assistant:", response)

Assistant: According to the memories, you like Python for programming.
